# Практическая работа №2. Часть 3Р
## Регуляризация: штраф на веса, прореживание, ранняя остановка

**Цель:** увидеть на опыте, как большая полносвязная сеть выучивает наизусть маленькую обучающую выборку вместе с ошибками разметки, написать своими руками штрафы L2 и L1, шаг AdamW, слой прореживания, раннюю остановку и сглаживание меток, сверить их с PyTorch и сравнить все способы в одной таблице.

> **Правило части.** В заданиях Р2 и Р3 штрафы и шаги оптимизатора пишутся на тензорах, параметр `weight_decay` готовых оптимизаторов из `torch.optim` нужен только для сверки. В Р4 слой прореживания пишется как наследник `nn.Module`, готовый `nn.Dropout` нужен только для сверки. В Р5 ранняя остановка пишется как собственный класс. В Р6 функция потерь со сглаживанием меток пишется на тензорах, готовая `F.cross_entropy(..., label_smoothing=...)` нужна только для сверки.

**Задача:** классификация одежды FashionMNIST, но обучающая выборка нарочно маленькая: 1000 картинок, от 86 до 115 на класс. Сеть `BigMLP` с двумя скрытыми слоями по 512 нейронов имеет 669 706 параметров, то есть почти 670 параметров на одну обучающую картинку. Выборка используется в двух вариантах: с правильными метками и с испорченными, где у каждой пятой картинки (200 из 1000) метка заменена на неверную, как бывает при ручной разметке. Валидационная (2000 картинок) и тестовая (2000 картинок) выборки не испорчены. Сквозные учебные примеры со слайдов: шаг со снижением весов для $w = (2, -1)$, первый шаг Adam со штрафом и AdamW для двух весов, маска прореживания для $h = (2, 4, 6, 8)$, ряд валидационных потерь для ранней остановки и сглаженные метки для четырех классов.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Там, где результат зависит от случайных чисел, указано примерное значение или качественное ожидание. Одно обучение сети на 1000 картинках за 60 эпох занимает на процессоре 10–20 секунд, всего в ноутбуке около 15 обучений. Сначала прочитайте теорию ниже.

## Как устроена регуляризация

**1. Емкость модели и переобучение.** Емкостью называют способность модели подстраиваться под самые разные зависимости (Гудфеллоу, раздел 5.2). У обучения две цели: сделать маленькой ошибку на обучающей выборке и сделать маленьким **разрыв** между ошибкой на обучающей выборке и ошибкой на новых данных. Если не достигнута первая цель, это недообучение; если вторая, это переобучение. Новые данные изображает валидационная выборка, а тестовая выборка используется один раз в самом конце (9 класс, Тема 1).

У сети этой работы 669 706 параметров и всего 1000 обучающих картинок, поэтому она может просто запомнить ответ для каждой картинки. Признаки переобучения на кривых обучения:
- потеря на обучающей выборке продолжает падать, а на валидационной после некоторой эпохи начинает расти;
- точность на обучающей выборке близка к 1, а на валидационной заметно ниже.

Почему валидационная потеря растет, даже если валидационная точность почти не меняется: перекрестная энтропия $-\ln p$ (здесь $p$ – вероятность, которую сеть дала правильному классу) наказывает за уверенность в неверном ответе. Переобученная сеть выдает вероятности, близкие к 0 и 1. На верно распознанной картинке она выигрывает почти ничего ($-\ln 0{,}999 = 0{,}001$), а каждая уверенная ошибка стоит дорого: при $p = 0{,}001$ потеря равна $6{,}9$, тогда как при равномерном ответе по 10 классам она равна $\ln 10 = 2{,}3$.

**2. Что такое регуляризация.** По определению Гудфеллоу (раздел 5.2.2 и введение к главе 7) регуляризация – любое изменение алгоритма обучения, которое должно уменьшить ошибку обобщения, а не ошибку обучения. Ошибка на обучающей выборке при этом обычно растет. Главная мысль главы 7: лучшая на практике модель – это большая модель с подходящей регуляризацией, а не маленькая модель «точно по размеру задачи». На этом занятии разбираются штраф на величину весов, прореживание, ранняя остановка и сглаживание меток. В Б2.4 к ним добавится аугментация данных, в Б2.5 – перенос знаний из предобученной сети.

**3. Штраф L2 и снижение весов.** К функции потерь $J$ прибавляют штраф за величину весов (раздел 7.1.1; у Гудфеллоу коэффициент штрафа обозначен $\alpha$, а скорость обучения $\varepsilon$):
$$\tilde J(w) = J(w) + \frac{\lambda}{2}\lVert w \rVert^2, \qquad \nabla \tilde J = \nabla J + \lambda w.$$
Шаг градиентного спуска со скоростью обучения $\eta$:
$$w \leftarrow w - \eta\,(\nabla J + \lambda w) = (1 - \eta\lambda)\,w - \eta\,\nabla J.$$
Перед обычным шагом каждый вес умножается на число чуть меньше 1, поэтому метод называют **снижением весов** (по-английски weight decay). Веса, которые не помогают уменьшать $J$, постепенно уходят к нулю.

*Пример со слайдов.* $w = (2;\ -1)$, градиент потери $\nabla J = (0{,}4;\ 0{,}2)$, $\eta = 0{,}1$, $\lambda = 0{,}5$. Множитель $1 - 0{,}1 \cdot 0{,}5 = 0{,}95$, новый вес $(0{,}95 \cdot 2 - 0{,}04;\ -0{,}95 - 0{,}02) = (1{,}86;\ -0{,}97)$. Без штрафа получилось бы $(1{,}96;\ -1{,}02)$. Значение штрафа в начальной точке $\frac{\lambda}{2}\lVert w\rVert^2 = 0{,}25 \cdot 5 = 1{,}25$.

*Что штраф делает с решением.* Около минимума $w^*$ функции потерь без штрафа ее можно приблизить квадратичной функцией с матрицей вторых производных $H$. Гудфеллоу показывает (формула 7.13), что минимум со штрафом получается из $w^*$ сжатием вдоль собственных векторов $H$: составляющая вдоль направления с собственным значением $h_i$ умножается на
$$\frac{h_i}{h_i + \lambda}.$$
При $\lambda = 0{,}1$ и собственных значениях $10;\ 1;\ 0{,}1;\ 0{,}01$ множители равны $0{,}99;\ 0{,}91;\ 0{,}5;\ 0{,}09$. Направления, вдоль которых потеря растет круто (данные настойчиво требуют определенного значения веса), почти не меняются. Пологие направления, о которых данные почти ничего не говорят, сжимаются к нулю, а именно в них сеть запоминает случайные особенности обучающей выборки.

*Смещения не штрафуют.* Вес связывает две переменные, а смещение относится к одной, поэтому для подбора смещения нужно меньше данных, и штраф на смещения почти не помогает, а может вызвать недообучение (раздел 7.1). Параметры нормализации из Б2.2 тоже обычно не штрафуют. В PyTorch для этого параметры делят на группы, как в Б2.1: у весов `weight_decay` равен $\lambda$, у смещений 0.

**4. Штраф L1.** $\tilde J = J + \lambda\lVert w\rVert_1 = J + \lambda\sum_i |w_i|$. Градиент штрафа $\lambda\,\mathrm{sign}(w)$: каждый вес толкается к нулю с одной и той же силой, каким бы маленьким он ни был. У штрафа L2 толчок $\lambda w$ слабеет вместе с весом, поэтому вес становится маленьким, но не нулевым. Для квадратичной функции потерь с диагональной матрицей $H$ решение со штрафом L1 записывается явно (формула 7.23):
$$w_i = \mathrm{sign}(w_i^*)\,\max\Big(|w_i^*| - \frac{\lambda}{H_{ii}},\ 0\Big).$$
Веса, у которых $|w_i^*| \le \lambda / H_{ii}$, становятся **ровно нулем**, остальные уменьшаются по модулю на $\lambda / H_{ii}$. Такое свойство называют разреженностью: штраф L1 сам отбирает нужные признаки. Пример: при $H_{ii} = 1$ и $\lambda = 0{,}5$ веса $w^* = (2;\ 0{,}3;\ -0{,}8)$ превращаются в $(1{,}5;\ 0;\ -0{,}3)$, а штраф L2 с тем же $\lambda$ дал бы $w^*/1{,}5 = (1{,}333;\ 0{,}2;\ -0{,}533)$. При обычном градиентном спуске вес со штрафом L1 колеблется около нуля, но точно нулем не становится; точные нули дает особый шаг, это задание повышенной сложности.

**5. Adam со штрафом L2 и AdamW.** Для SGD прибавить $\lambda w$ к градиенту и умножить вес на $(1 - \eta\lambda)$ – одно и то же. Для Adam это не так. Если прибавить $\lambda w$ к градиенту (так работает `torch.optim.Adam(..., weight_decay=λ)`), штраф попадает в скользящие средние $m$ и $v$ и делится на $\sqrt{\hat v}$ вместе с градиентом. У весов с большими градиентами $\sqrt{\hat v}$ велико, и штраф почти не действует; у весов с маленькими градиентами штраф, наоборот, усиливается. Лощилов и Хуттер (2019) предложили отделить снижение весов от градиента:
$$w \leftarrow w - \eta\left(\frac{\hat m}{\sqrt{\hat v} + \varepsilon} + \lambda w\right),$$
это **AdamW** (`torch.optim.AdamW`): за шаг каждый вес уменьшается в одно и то же число раз $1 - \eta\lambda$, а $\hat m$ и $\hat v$ считаются только по градиенту потери.

*Пример со слайдов.* Два веса $w = (1;\ 1)$, градиент потери $g = (1;\ 0)$, $\eta = 0{,}1$, $\lambda = 0{,}1$, первый шаг. На первом шаге у Adam $\hat m = g$ и $\hat v = g^2$, поэтому шаг равен $\eta \cdot g/|g|$, то есть $\pm\eta$ для любого ненулевого градиента (Б2.1).
- Adam со штрафом в градиенте: $g + \lambda w = (1{,}1;\ 0{,}1)$, оба числа положительны, шаг $(0{,}1;\ 0{,}1)$, новый вес $(0{,}9;\ 0{,}9)$. Первый вес изменился так же, как без штрафа, а второй вес, у которого градиент потери нулевой, уменьшился на $0{,}1$ – в 10 раз сильнее задуманного снижения $\eta\lambda w = 0{,}01$.
- AdamW: шаг Adam по градиенту $g$ равен $(0{,}1;\ 0)$, снижение $\eta\lambda w = (0{,}01;\ 0{,}01)$, новый вес $(0{,}89;\ 0{,}99)$: снижение у обоих весов одинаковое, 1 %.

Обратите внимание: у `torch.optim.AdamW` значение `weight_decay` по умолчанию 0,01 и умножается на скорость обучения, поэтому при $\eta = 10^{-3}$ заметное действие дают значения порядка 1–10.

**6. Прореживание.** Прореживание (по-английски dropout; Сривастава, Хинтон и соавторы, 2014) работает так: во время обучения на каждом шаге каждый нейрон слоя независимо от других **выключается** с вероятностью $p$ (его выход заменяется нулем), а выходы оставшихся нейронов умножаются на $\frac{1}{1-p}$:
$$\tilde h_i = \frac{\mu_i}{1-p}\,h_i, \qquad \mu_i = \begin{cases} 1 & \text{с вероятностью } 1 - p,\\ 0 & \text{с вероятностью } p.\end{cases}$$
Множитель сохраняет среднее значение выхода: $\mathbb{E}[\tilde h_i] = (1-p)\cdot\frac{h_i}{1-p} = h_i$. При применении сети прореживание выключено, слой пропускает значения без изменений. Здесь принята договоренность PyTorch: в `nn.Dropout(p)` число $p$ – вероятность **выключить** нейрон. У Гудфеллоу говорится о вероятности включения, она равна $1 - p$.

*Пример со слайдов.* $h = (2, 4, 6, 8)$, $p = 0{,}25$, выпала маска $\mu = (1, 0, 1, 1)$. Множитель $1/0{,}75 = 4/3$, выход $(2{,}667;\ 0;\ 8;\ 10{,}667)$. Сумма выхода $21{,}33$ вместо $20$: на одном шаге она случайна, а в среднем по всем маскам равна $20$.

*Почему это помогает.*
- Нейрон не может рассчитывать на определенных соседей: любой из них может пропасть. Признаки должны быть полезны сами по себе, а сложная совместная подстройка нейронов, которой сеть запоминает обучающую выборку, разрушается.
- Ансамбль: каждая маска задает подсеть, и все подсети пользуются общими весами. В нашей сети $512 + 512 = 1024$ нейрона, которые можно выключать, то есть $2^{1024} \approx 10^{308}$ подсетей, и каждая обучается на своих мини-пакетах. При применении полная сеть приближенно заменяет усреднение всех подсетей; у Гудфеллоу это называется правилом вывода с масштабированием весов (раздел 7.12). Обычный ансамбль из 5–10 сетей (баггинг, раздел 7.11) требует в 5–10 раз больше вычислений.

*Где ставить и какое $p$ выбирать.* После функции активации полносвязного слоя; часто $p = 0{,}5$ для скрытых слоев и $p = 0{,}2$ для входа. В сверточных сетях прореживание ставят реже, обычно перед полносвязной частью, а сверточные слои регуляризует пакетная нормализация. Режимы такие же, как у пакетной нормализации: `model.train()` и `model.eval()`. Если забыть `model.eval()`, ответы сети на одну и ту же картинку станут случайными.

**7. Ранняя остановка.** После каждой эпохи считают потерю на валидационной выборке и запоминают веса с лучшим значением. Если $P$ эпох подряд лучше не стало, обучение останавливают и возвращают лучшие веса (алгоритм 7.1). Число $P$ называют терпением.

*Пример со слайдов.* Валидационные потери по эпохам: $0{,}90;\ 0{,}70;\ 0{,}62;\ 0{,}60;\ 0{,}61;\ 0{,}63;\ 0{,}59;\ 0{,}64;\ 0{,}66;\ 0{,}70$. При терпении 2 обучение остановится после эпохи 6 с весами эпохи 4 (потеря $0{,}60$) и не увидит улучшения до $0{,}59$ в эпохе 7. При терпении 3 обучение остановится после эпохи 10 с весами эпохи 7. Большое терпение стоит лишнего времени, маленькое рискует остановиться на случайном подъеме кривой.

*Ранняя остановка как штраф.* Пока сеть сделала немного шагов, веса не успевают уйти далеко от маленьких начальных значений. Для квадратичной функции потерь и градиентного спуска Гудфеллоу показывает (раздел 7.8, формулы 7.40–7.45), что $\tau$ шагов со скоростью обучения $\eta$ действуют как штраф L2 с коэффициентом
$$\lambda \approx \frac{1}{\eta\,\tau}.$$
Например, $\eta = 0{,}1$ и $\tau = 100$ шагов соответствуют $\lambda \approx 0{,}1$. Преимущество перед штрафом: силу регуляризации подбирают за одно обучение, по одной кривой. Недостаток: нужна валидационная выборка. Так как по ней выбирают эпоху, она становится частью обучения, и честную итоговую оценку дает только тестовая выборка.

**8. Сглаживание меток.** Если целевые вероятности равны ровно 0 и 1, перекрестная энтропия достигает минимума только при вероятности правильного класса ровно 1, то есть при бесконечно большом разрыве между логитами. Сеть будет без конца увеличивать веса и делать все более уверенные предсказания (раздел 7.5.1). Сглаживание меток заменяет целевое распределение:
$$q_k = (1 - \varepsilon)\,[k = y] + \frac{\varepsilon}{K}, \qquad L = -\sum_{k=1}^{K} q_k \ln p_k,$$
где $K$ – число классов, $\varepsilon$ – сила сглаживания, $[k = y]$ равно 1 для правильного класса и 0 для остальных. Пример: $K = 4$, $\varepsilon = 0{,}1$: правильный класс получает $0{,}9 + 0{,}025 = 0{,}925$, остальные по $0{,}025$. Минимум потери достигается при $p = q$, то есть при разрыве логитов $\ln(0{,}925/0{,}025) = \ln 37 \approx 3{,}61$, а не бесконечном. Для логитов $(2, 0, 0, 0)$ и правильного класса 0 обычная потеря равна $0{,}341$, сглаженная $0{,}491$. У Гудфеллоу описан вариант с метками $1 - \varepsilon$ и $\varepsilon/(K-1)$; в PyTorch (`F.cross_entropy(..., label_smoothing=ε)`) используется формула выше.

**9. Ошибки в разметке.** Во втором варианте обучающей выборки у каждой пятой картинки метка заменена на неверную. Сеть, которая выучила обучающую выборку наизусть, выучит и эти 200 ошибок, а такие «правила» испортят ответы на похожих новых картинках. Удобно измерять две доли на испорченных картинках: доля, где сеть отвечает испорченной меткой (запомнила ошибку), и доля, где она отвечает настоящей меткой (не поверила разметке). Отсюда же следует, что точность на обучающей выборке с испорченными метками выше $0{,}8$ – плохой признак: сеть, которая всегда отвечает правильно, получит на ней ровно $0{,}8$.

**10. В PyTorch.**
- `torch.optim.AdamW([{'params': веса, 'weight_decay': λ}, {'params': смещения, 'weight_decay': 0.0}], lr=...)` – снижение весов только для весов.
- `nn.Dropout(p)` – слой прореживания, режим переключают `model.train()` и `model.eval()`.
- `F.cross_entropy(logits, y, label_smoothing=0.1)` – сглаживание меток.
- Ранняя остановка пишется вручную: `copy.deepcopy(model.state_dict())` сохраняет копию весов, `model.load_state_dict(...)` возвращает их. Без `deepcopy` словарь `state_dict()` содержит ссылки на те же тензоры, которые продолжат меняться при обучении.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import copy
import math
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

# Цвета графиков – как на слайдах
NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# FashionMNIST заранее скачивается в папку data (интернет в аудитории не нужен):
#   datasets.FashionMNIST('data', train=True, download=True); datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
tf = transforms.ToTensor()
try:
    train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=False, transform=tf)
except Exception:
    print('FashionMNIST не найден в папке data – используются случайные картинки FakeData, ожидаемые числа не совпадут')
    train_ds = datasets.FakeData(size=60000, image_size=(1, 28, 28), num_classes=10, transform=tf)
    test_ds = datasets.FakeData(size=2000, image_size=(1, 28, 28), num_classes=10, transform=tf, random_offset=60000)

def take(ds, idx):
    """Картинки и метки с номерами idx одним тензором: выборка маленькая и целиком помещается в память."""
    idx = list(idx)
    if hasattr(ds, 'data'):
        return ds.data[idx].float().div(255).unsqueeze(1), ds.targets[idx].clone()
    xs, ys = zip(*(ds[i] for i in idx))
    return torch.stack(xs), torch.as_tensor([int(y) for y in ys])

X_train, y_clean = take(train_ds, range(1000))            # обучающая выборка: первые 1000 картинок
X_val, y_val = take(train_ds, range(58000, 60000))        # валидационная: последние 2000 картинок обучающего набора
X_test, y_test = take(test_ds, range(2000))               # тестовая: первые 2000 картинок тестового набора

# Испорченная разметка: у каждой пятой картинки метка сдвинута на 1–9 классов, то есть всегда неверная
noisy_idx = torch.arange(0, 1000, 5)
y_noisy = y_clean.clone()
y_noisy[noisy_idx] = (y_clean[noisy_idx] + 1 + torch.arange(len(noisy_idx)) % 9) % 10
print(tuple(X_train.shape), tuple(X_val.shape), tuple(X_test.shape), '  ожидается (1000, 1, 28, 28) (2000, 1, 28, 28) (2000, 1, 28, 28)')
print(torch.bincount(y_clean, minlength=10).tolist(), (y_noisy != y_clean).sum().item(),
      '  ожидается [107, 104, 86, 92, 95, 100, 100, 115, 102, 99] 200')

# Большая полносвязная сеть: 784 -> 512 -> 512 -> 10, после каждого скрытого слоя ReLU и прореживание
class BigMLP(nn.Module):
    def __init__(self, p=0.0, dropout_cls=nn.Dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(), dropout_cls(p),
            nn.Linear(512, 512), nn.ReLU(), dropout_cls(p),
            nn.Linear(512, 10))

    def forward(self, x):
        return self.net(x)

count = lambda m: sum(p.numel() for p in m.parameters())
print(count(BigMLP()), '  ожидается 669706')

def make_adamw(model, wd, lr=1e-3):
    """AdamW со снижением только весов (тензоры с dim > 1); при wd = 0 это обычный Adam."""
    decay = [p for p in model.parameters() if p.dim() > 1]
    no_decay = [p for p in model.parameters() if p.dim() <= 1]
    return torch.optim.AdamW([{'params': decay, 'weight_decay': wd},
                              {'params': no_decay, 'weight_decay': 0.0}], lr=lr)

def fit(model, opt, y_train, epochs=60, batch_size=50, loss_fn=F.cross_entropy, extra_loss=None, stopper=None):
    """Обучение по мини-пакетам на X_train с метками y_train; после каждой эпохи – оценка на обучающей и валидационной выборках.
    extra_loss(model) – штраф, который прибавляется к потере; stopper – ранняя остановка из задания Р5."""
    g = torch.Generator().manual_seed(RANDOM_STATE)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(X_train), generator=g)
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X_train[idx]), y_train[idx])
            if extra_loss is not None:
                loss = loss + extra_loss(model)
            loss.backward()
            opt.step()
        tl, ta = evaluate(model, X_train, y_train)          # evaluate пишется в задании Р1
        vl, va = evaluate(model, X_val, y_val)
        for key, value in zip(hist, [tl, ta, vl, va]):
            hist[key].append(value)
        if stopper is not None and stopper.step(vl, model, epoch):
            break
    if stopper is not None:
        stopper.restore(model)
    return hist

results, models = {}, {}

def train_variant(name, y_train, p=0.0, wd=0.0, epochs=60, dropout_cls=nn.Dropout, **fit_kwargs):
    """Одно обучение BigMLP с одними и теми же начальными весами; итоги складываются в results и models."""
    start = time.perf_counter()
    torch.manual_seed(RANDOM_STATE)
    model = BigMLP(p, dropout_cls)
    hist = fit(model, make_adamw(model, wd), y_train, epochs=epochs, **fit_kwargs)
    tl, ta = evaluate(model, X_train, y_train)
    vl, va = evaluate(model, X_val, y_val)
    _, te = evaluate(model, X_test, y_test)
    mem, fix = memorized_share(model, X_train, y_noisy, y_clean, noisy_idx)   # пишется в задании Р1
    results[name] = dict(train_acc=ta, val_acc=va, val_loss=vl, test_acc=te, gap=ta - va,
                         mem=mem, fix=fix, noisy=y_train is y_noisy, epochs=len(hist['val_loss']), hist=hist)
    models[name] = model
    tail = f'   запомнено ошибок {mem:.2f}' if y_train is y_noisy else ''
    print(f'{name:<24} эпох {len(hist["val_loss"]):>3}   обуч. {ta:.3f}   валид. {va:.3f} (потеря {vl:.3f})'
          f'   тест {te:.3f}{tail}   [{time.perf_counter() - start:.0f} с]')
    return model

def plot_runs(names, fname, title=''):
    """Кривые обучения: слева потеря, справа точность; пунктир – обучающая выборка, сплошная – валидационная."""
    fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
    for name, c in zip(names, [GRAY, NAVY, DEEP, BLUE]):
        h = results[name]['hist']
        ep = range(1, len(h['val_loss']) + 1)
        axs[0].plot(ep, h['train_loss'], color=c, lw=1.4, ls='--')
        axs[0].plot(ep, h['val_loss'], color=c, lw=2.2, label=name)
        axs[1].plot(ep, h['train_acc'], color=c, lw=1.4, ls='--')
        axs[1].plot(ep, h['val_acc'], color=c, lw=2.2, label=name)
    axs[0].set_ylabel('потеря'); axs[1].set_ylabel('точность')
    for ax in axs:
        ax.set_xlabel('эпоха')
    axs[0].legend(fontsize=9)
    axs[0].set_title(title or 'сплошная – валидационная выборка, пунктир – обучающая', fontsize=10)
    fig.tight_layout(); fig.savefig(fname, dpi=100); plt.close(fig)
    display(Image(fname))

**Задание Р1. Переобучение на 1000 картинках.**

- `evaluate(model, X, y)` переводит модель в режим применения (`model.eval()`), внутри `torch.no_grad()` считает логиты для всех объектов сразу и возвращает пару чисел: среднюю перекрестную энтропию `F.cross_entropy(logits, y).item()` и долю верных ответов (`argmax` по классам сравнивается с `y`).
- `memorized_share(model, X, y_noisy, y_clean, idx)` считает ответы модели на картинках с номерами `idx` (это испорченные картинки) и возвращает две доли: сколько ответов совпало с испорченной меткой и сколько с настоящей (п. 9 теории).

Обе функции нужны готовым функциям `fit` и `train_variant` из ячейки подготовки.

*Проверка.* «Модель», которая на любую картинку выдает нулевые логиты, получает потерю $\ln 10 \approx 2{,}303$ и всегда отвечает классом 0 (у `argmax` при равных значениях выигрывает первый номер), а класс 0 составляет 9,6 % валидационной выборки. «Модель», которая отвечает ровно испорченными метками, запомнила все ошибки: доли $(1{,}0;\ 0{,}0)$. Затем готовая ячейка обучает `BigMLP` без регуляризации 60 эпох на правильных и на испорченных метках.

In [ ]:
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
    loss = ...                                      # <-- средняя перекрестная энтропия, число
    acc = ...                                       # <-- доля верных ответов
    return loss, acc

def memorized_share(model, X, y_noisy, y_clean, idx):
    model.eval()
    with torch.no_grad():
        pred = model(X[idx]).argmax(dim=1)
    mem = ...                                       # <-- доля ответов, совпавших с испорченной меткой
    fix = ...                                       # <-- доля ответов, совпавших с настоящей меткой
    return mem, fix


class ZeroModel(nn.Module):              # «модель» без параметров: нулевые логиты на любую картинку
    def forward(self, x):
        return torch.zeros(len(x), 10)

class Answers(nn.Module):                # «модель», которая выдает заранее заданные ответы
    def __init__(self, answers):
        super().__init__()
        self.answers = answers
    def forward(self, x):
        return F.one_hot(self.answers[:len(x)], 10).float()

loss0, acc0 = evaluate(ZeroModel(), X_val, y_val)
print(round(loss0, 3), round(acc0, 3), '  ожидается 2.303 0.096')
print(memorized_share(Answers(y_noisy[noisy_idx]), X_train, y_noisy, y_clean, noisy_idx), '  ожидается (1.0, 0.0)')

In [ ]:
# Готовая ячейка: BigMLP без регуляризации, Adam (lr = 0.001), 60 эпох, пакеты по 50 картинок
train_variant('без регуляризации', y_clean)
train_variant('метки с ошибками', y_noisy)
h = results['без регуляризации']['hist']
print('эпоха минимума валидационной потери:', int(np.argmin(h['val_loss'])) + 1)
print('  ожидается: на правильных метках обучение около 1.00, валидация около 0.81, потеря на валидации в конце около 1.0;')
print('  минимум валидационной потери около 5–10 эпохи (около 0.57); на испорченных метках обучение около 0.99,')
print('  валидация около 0.71, запомнено больше 0.9 испорченных меток')
plot_runs(['без регуляризации', 'метки с ошибками'], 'overfit_curves.png')

**Вопрос Р1.** а) В какой эпохе валидационная потеря минимальна? Почему после этого она растет, хотя валидационная точность почти не меняется? б) Почему точность около 0,99 на обучающей выборке с испорченными метками – плохой результат? Какую точность на ней получила бы сеть, которая всегда отвечает правильно? в) Сколько параметров приходится на одну обучающую картинку и почему сеть может запомнить 1000 ответов, включая неверные?

*Ваш ответ:*

**Задание Р2. Штрафы L2 и L1.**

- `l2_penalty(model)` – половина суммы квадратов **весов** модели: $\frac12\sum \lVert W\rVert^2$ по всем параметрам с `p.dim() > 1`. Смещения (одномерные тензоры) не штрафуются (п. 3 теории).
- `l1_penalty(model)` – сумма модулей весов по тем же параметрам (п. 4).
- `decay_step_(params, lr, wd)` – шаг SGD со снижением весов на месте, внутри `torch.no_grad()`: сначала `p.mul_(1 - lr * wd)`, потом `p.sub_(lr * p.grad)`.

*Проверка по п. 3 теории.* У слоя `nn.Linear(2, 1)` с весами $(2, -1)$ и смещением 3 штраф L2 равен $2{,}5$, штраф L1 равен 3. Шаг для $w = (2, -1)$, $\nabla J = (0{,}4;\ 0{,}2)$, $\eta = 0{,}1$, $\lambda = 0{,}5$ дает $(1{,}86;\ -0{,}97)$, как и `torch.optim.SGD(..., weight_decay=0.5)`. Затем готовая ячейка обучает сеть на испорченных метках со снижением весов (AdamW, $\lambda = 1, 5, 10$) и со штрафом L1 и показывает карты весов первого слоя.

In [ ]:
def l2_penalty(model):
    """Половина суммы квадратов всех весов (тензоров с dim > 1); смещения не штрафуются."""
    return ...                                      # <-- 0.5 * сумма квадратов весов

def l1_penalty(model):
    """Сумма модулей всех весов (тензоров с dim > 1)."""
    return ...                                      # <-- сумма модулей весов

def decay_step_(params, lr, wd):
    """Шаг SGD со снижением весов: w <- (1 - lr * wd) * w - lr * grad."""
    with torch.no_grad():
        for p in params:
            ...                                     # <-- умножить вес на (1 - lr * wd) на месте
            ...                                     # <-- обычный шаг против градиента на месте


lin = nn.Linear(2, 1)
with torch.no_grad():
    lin.weight.copy_(torch.tensor([[2.0, -1.0]]))
    lin.bias.fill_(3.0)
print(show(l2_penalty(lin)), show(l1_penalty(lin)), '  ожидается 2.5 3.0 (смещение 3 в штраф не входит)')

w = torch.tensor([2.0, -1.0], requires_grad=True)
w.grad = torch.tensor([0.4, 0.2])
decay_step_([w], lr=0.1, wd=0.5)
w_ref = torch.tensor([2.0, -1.0], requires_grad=True)
w_ref.grad = torch.tensor([0.4, 0.2])
torch.optim.SGD([w_ref], lr=0.1, weight_decay=0.5).step()
print(show(w), show(w_ref), '  ожидается [1.86, -0.97] [1.86, -0.97]')

# Штраф в функции потерь дает тот же градиент, что и weight_decay: (0.4, 0.2) + 0.5 * (2, -1)
w2 = torch.tensor([2.0, -1.0], requires_grad=True)
(torch.tensor([0.4, 0.2]) * w2).sum().add(0.5 * 0.5 * (w2 ** 2).sum()).backward()
print(show(w2.grad), '  ожидается [1.4, -0.3]')

In [ ]:
# Готовая ячейка: снижение весов (AdamW) и штраф L1 на испорченных метках
for wd in [1, 5, 10]:
    train_variant(f'снижение весов {wd}', y_noisy, wd=wd)
train_variant('штраф L1 0.001', y_noisy, extra_loss=lambda m: 1e-3 * l1_penalty(m))
print('  ожидается: λ = 1 почти ничего не меняет; λ = 5 и λ = 10 – валидация около 0.74–0.76, запомнено меньше 0.3 ошибок;')
print('  штраф L1 – валидация около 0.75, запомнено меньше 0.1 ошибок')
plot_runs(['метки с ошибками', 'снижение весов 1', 'снижение весов 5', 'снижение весов 10'], 'weight_decay_curves.png')

fig, axs = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, name in zip(axs, ['метки с ошибками', 'снижение весов 10', 'штраф L1 0.001']):
    W1 = models[name].net[1].weight.detach()                    # веса первого слоя: 512 x 784
    near_zero = (W1.abs() < 1e-3).float().mean().item()
    ax.imshow(W1.abs().mean(dim=0).view(28, 28), cmap='Blues')   # средний модуль веса для каждого пикселя
    ax.set_title(f'{name}\nвесов меньше 0.001: {near_zero:.0%}', fontsize=10)
    ax.axis('off')
fig.tight_layout(); fig.savefig('weight_maps.png', dpi=100); plt.close(fig)
display(Image('weight_maps.png'))
print('  ожидается (у каждой карты своя шкала цвета): без штрафа самые крупные веса у краевых пикселей, малых весов около 2 %;')
print('  со снижением весов все веса в 3–5 раз меньше, малых около 20 %; со штрафом L1 малых весов около 99 %,')
print('  а заметные веса остались в основном в центре картинки, где находится одежда')

**Вопрос Р2.** а) Почему при $\lambda = 1$ в AdamW снижение почти не действует? Во сколько раз уменьшился бы вес за все обучение без градиента потери ($\eta = 10^{-3}$, 60 эпох по 20 шагов)? б) Почему без штрафа самые крупные веса первого слоя оказались у пикселей на краю картинки, которые почти всегда черные? Что с ними делают штрафы L2 и L1? в) Почему штраф L1 оставил почти все веса первого слоя близкими к нулю, а точность на валидации не упала?

*Ваш ответ:*

**Задание Р3. Adam со штрафом L2 и AdamW.**

Готовая функция `adam_direction(g, state)` (как в Б2.1) обновляет моменты `m`, `v` и номер шага `t` в словаре `state` и возвращает направление $\hat m / (\sqrt{\hat v} + \varepsilon)$. Напишите два шага по п. 5 теории, каждая функция возвращает новый вес:
- `adam_l2_step(w, g, state, lr, wd)` – штраф прибавляется к градиенту: в `adam_direction` передается `g + wd * w`, новый вес `w - lr * направление`;
- `adamw_step(w, g, state, lr, wd)` – в `adam_direction` передается сам `g`, а снижение `lr * wd * w` вычитается отдельно.

*Проверка по п. 5 теории.* $w = (1;\ 1)$, $g = (1;\ 0)$, $\eta = 0{,}1$, $\lambda = 0{,}1$, первый шаг: Adam со штрафом дает $(0{,}9;\ 0{,}9)$, AdamW дает $(0{,}89;\ 0{,}99)$. Те же числа должны получиться у `torch.optim.Adam(..., weight_decay=0.1)` и `torch.optim.AdamW(..., weight_decay=0.1)`.

In [ ]:
def new_state(w):
    return {'t': 0, 'm': torch.zeros_like(w), 'v': torch.zeros_like(w)}

def adam_direction(g, state, beta1=0.9, beta2=0.999, eps=1e-8):
    """Готовая функция из Б2.1: обновляет моменты в state и возвращает m_hat / (sqrt(v_hat) + eps)."""
    state['t'] += 1
    state['m'] = beta1 * state['m'] + (1 - beta1) * g
    state['v'] = beta2 * state['v'] + (1 - beta2) * g ** 2
    m_hat = state['m'] / (1 - beta1 ** state['t'])
    v_hat = state['v'] / (1 - beta2 ** state['t'])
    return m_hat / (v_hat.sqrt() + eps)

def adam_l2_step(w, g, state, lr, wd):
    """Adam со штрафом L2: wd * w прибавляется к градиенту и проходит через нормировку."""
    return ...                                      # <-- направление по градиенту со штрафом

def adamw_step(w, g, state, lr, wd):
    """AdamW: шаг Adam по градиенту потери и отдельное снижение веса."""
    return ...                                      # <-- направление по g и снижение lr * wd * w


w0, g0 = torch.tensor([1.0, 1.0]), torch.tensor([1.0, 0.0])
print(show(adam_l2_step(w0, g0, new_state(w0), 0.1, 0.1)), show(adamw_step(w0, g0, new_state(w0), 0.1, 0.1)),
      '  ожидается [0.9, 0.9] [0.89, 0.99]')

def torch_first_step(opt_cls):
    w = torch.tensor([1.0, 1.0], requires_grad=True)
    w.grad = torch.tensor([1.0, 0.0])
    opt_cls([w], lr=0.1, weight_decay=0.1).step()
    return w
print(show(torch_first_step(torch.optim.Adam)), show(torch_first_step(torch.optim.AdamW)), '  ожидается [0.9, 0.9] [0.89, 0.99]')

# Готовая часть: 20 шагов для веса, у которого градиент потери всегда нулевой
for label, step_fn in [('Adam со штрафом', adam_l2_step), ('AdamW', adamw_step)]:
    for wd in [0.01, 0.1]:
        w, st = torch.tensor([1.0]), new_state(torch.tensor([1.0]))
        for _ in range(20):
            w = step_fn(w, torch.zeros(1), st, 0.1, wd)
        print(f'{label:<16} λ = {wd:<5} вес после 20 шагов: {w.item():.3f}')
print('  ожидается: у Adam со штрафом вес за 20 шагов проскакивает через ноль (около -0.271) одинаково при λ = 0.01 и λ = 0.1;')
print('  у AdamW вес равен (1 - 0.1·λ) в степени 20: 0.980 и 0.818')

**Вопрос Р3.** а) Почему у Adam со штрафом в градиенте вес без собственного градиента уменьшается почти одинаково при $\lambda = 0{,}01$ и $\lambda = 0{,}1$? б) Почему для SGD разницы между штрафом в градиенте и отдельным снижением весов нет? в) У какого из двух методов коэффициент $\lambda$ проще подбирать и переносить из одной задачи в другую?

*Ваш ответ:*

**Задание Р4. Свой слой прореживания (главное задание).**

- `dropout_with_mask(x, mask, p)` – выход по п. 6 теории при известной маске: `x * mask / (1 - p)`.
- Класс `MyDropout(p)` – наследник `nn.Module`. В режиме применения (`self.training` равно `False`) или при `p = 0` слой возвращает вход без изменений. В режиме обучения он создает маску `(torch.rand_like(x) >= p).float()` (единица с вероятностью $1 - p$) и возвращает `dropout_with_mask(x, mask, p)`. Случай `p = 1` уже написан: все нейроны выключены, выход нулевой.

*Проверка по п. 6 теории.* $h = (2, 4, 6, 8)$, маска $(1, 0, 1, 1)$, $p = 0{,}25$: выход $(2{,}667;\ 0;\ 8;\ 10{,}667)$. На 100 000 единицах доля нулей около $0{,}25$, среднее около 1, градиент по входу принимает два значения: 0 и $4/3$. В режиме применения выход равен входу. Затем ваш слой встает в `BigMLP` вместо `nn.Dropout`, и сеть обучается на испорченных метках с $p = 0{,}2;\ 0{,}5;\ 0{,}7$.

In [ ]:
def dropout_with_mask(x, mask, p):
    return ...                                      # <-- обнулить по маске, оставшиеся умножить на 1 / (1 - p)

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return ...                              # <-- режим применения: вход без изменений
        if self.p == 1:
            return torch.zeros_like(x)
        mask = ...                                  # <-- 1 с вероятностью 1 - p, 0 с вероятностью p
        return ...                                  # <-- применить маску


h = torch.tensor([2.0, 4.0, 6.0, 8.0])
print(show(dropout_with_mask(h, torch.tensor([1.0, 0.0, 1.0, 1.0]), 0.25)), '  ожидается [2.667, 0.0, 8.0, 10.667]')

torch.manual_seed(RANDOM_STATE)
drop = MyDropout(0.25)
ones = torch.ones(100_000, requires_grad=True)
out = drop(ones)
out.sum().backward()
print(round((out == 0).float().mean().item(), 2), round(out.mean().item(), 2), sorted(set(show(ones.grad))),
      '  ожидается 0.25 1.0 [0.0, 1.333]')
drop.eval()
print(torch.equal(drop(h), h), '  ожидается True (в режиме применения слой ничего не меняет)')
ref = nn.Dropout(0.25)
r = ref(torch.ones(100_000))
print(round((r == 0).float().mean().item(), 2), round(r.mean().item(), 2), '  ожидается 0.25 1.0 (готовый слой PyTorch)')

In [ ]:
# Готовая ячейка: BigMLP с вашим слоем MyDropout на испорченных метках
for p in [0.2, 0.5, 0.7]:
    train_variant(f'прореживание {p}', y_noisy, p=p, dropout_cls=MyDropout)
print('  ожидается: чем больше p, тем ниже точность на обучающей выборке и тем меньше запомнено ошибок;')
print('  при p = 0.7 валидация около 0.78, запомнено около 0.2 ошибок')
plot_runs(['метки с ошибками', 'прореживание 0.2', 'прореживание 0.5', 'прореживание 0.7'], 'dropout_curves.png')

net = models['прореживание 0.5']
x1 = X_val[:1]
for mode in ['train', 'eval']:
    getattr(net, mode)()
    with torch.no_grad():
        probs = [F.softmax(net(x1), dim=1)[0, y_val[0]].item() for _ in range(5)]
    print(f'{mode}: вероятность правильного класса при пяти вызовах', [round(v, 3) for v in probs])
print('  ожидается: в режиме train пять разных чисел, в режиме eval пять одинаковых')

**Вопрос Р4.** а) Зачем выход умножается на $1/(1-p)$? Что случилось бы в режиме применения, если бы множителя не было, а прореживание по-прежнему выключалось? б) Почему при $p = 0{,}7$ точность на обучающей выборке ниже, чем на валидационной? в) Почему прореживание называют дешевым ансамблем? Сколько подсетей в нашей сети и сколько из них обучено хотя бы на одном шаге?

*Ваш ответ:*

**Задание Р5. Ранняя остановка.**

Допишите класс `EarlyStopping(patience, min_delta=0.0)` по п. 7 теории. Метод `step(val_loss, model, epoch)` вызывается после каждой эпохи и возвращает `True`, если пора остановиться:
- если `val_loss < best_loss - min_delta`, это улучшение: запомнить потерю, номер эпохи и **копию** весов `copy.deepcopy(model.state_dict())`, обнулить счетчик эпох без улучшения;
- иначе увеличить счетчик на 1;
- вернуть `True`, когда счетчик дошел до `patience`.

Метод `restore(model)` загружает сохраненные веса: `model.load_state_dict(self.best_state)`. Функция `fit` из ячейки подготовки уже умеет вызывать оба метода.

*Проверка по п. 7 теории.* Ряд потерь $0{,}90;\ 0{,}70;\ 0{,}62;\ 0{,}60;\ 0{,}61;\ 0{,}63;\ 0{,}59;\ 0{,}64;\ 0{,}66;\ 0{,}70$ подается вместе с «моделью» `nn.Linear(1, 1)`, у которой вес после эпохи равен номеру эпохи. При терпении 2 остановка после эпохи 6, лучшая эпоха 4, восстановленный вес 4; при терпении 3 остановка после эпохи 10, лучшая эпоха 7, вес 7. Если веса сохраняются без `deepcopy`, восстановленный вес окажется номером последней эпохи. Затем готовая ячейка обучает сеть на испорченных метках с терпением 5.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = float('inf')
        self.best_epoch = 0
        self.best_state = None
        self.bad_epochs = 0                                   # сколько эпох подряд без улучшения

    def step(self, val_loss, model, epoch):
        """Вызывается после каждой эпохи; возвращает True, если пора остановиться."""
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = ...                    # <-- новое лучшее значение потери
            self.best_epoch = ...                   # <-- номер лучшей эпохи
            self.best_state = ...                   # <-- копия весов, а не ссылка на них
            self.bad_epochs = ...                   # <-- обнулить счетчик
        else:
            ...                                     # <-- еще одна эпоха без улучшения
        return ...                                  # <-- пора ли остановиться

    def restore(self, model):
        ...                                         # <-- вернуть лучшие веса


val_losses = [0.90, 0.70, 0.62, 0.60, 0.61, 0.63, 0.59, 0.64, 0.66, 0.70]
for patience in [2, 3]:
    dummy = nn.Linear(1, 1)
    stopper = EarlyStopping(patience)
    for epoch, vl in enumerate(val_losses, start=1):
        with torch.no_grad():
            dummy.weight.fill_(epoch)                         # «веса» после эпохи равны ее номеру
        if stopper.step(vl, dummy, epoch):
            break
    stopper.restore(dummy)
    print(f'терпение {patience}: остановка после эпохи {epoch}, лучшая эпоха {stopper.best_epoch}, '
          f'потеря {stopper.best_loss}, восстановленный вес {dummy.weight.item():.0f}')
print('  ожидается: терпение 2 – эпохи 6 и 4, потеря 0.6, вес 4; терпение 3 – эпохи 10 и 7, потеря 0.59, вес 7')

In [ ]:
# Готовая ячейка: ранняя остановка с терпением 5 на испорченных метках (не больше 60 эпох)
stopper = EarlyStopping(patience=5)
train_variant('ранняя остановка', y_noisy, stopper=stopper)
print('лучшая эпоха:', stopper.best_epoch)
print('  ожидается: остановка примерно после 12–16 эпохи, лучшая эпоха около 7–11; валидация около 0.78,')
print('  запомнено меньше 0.2 ошибок – почти как у лучших штрафов, но за 15 эпох вместо 60')

fig, ax = plt.subplots(figsize=(7, 3.4))
full = results['метки с ошибками']['hist']['val_loss']
stopped = results['ранняя остановка']['hist']['val_loss']
ax.plot(range(1, len(full) + 1), full, color=GRAY, lw=2, label='без остановки')
ax.plot(range(1, len(stopped) + 1), stopped, color=BLUE, lw=2.6, label='с ранней остановкой')
ax.axvline(stopper.best_epoch, color=NAVY, ls='--', lw=1.2)
ax.text(stopper.best_epoch + 1, max(full) * 0.9, f'лучшая эпоха {stopper.best_epoch}', color=NAVY, fontsize=9)
ax.set_xlabel('эпоха'); ax.set_ylabel('потеря на валидации'); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig('early_stopping.png', dpi=110); plt.close(fig)
display(Image('early_stopping.png'))

**Вопрос Р5.** а) Что произойдет, если в `step` сохранять `model.state_dict()` без `copy.deepcopy`? б) Почему после ранней остановки точность на валидационной выборке уже нельзя считать честной оценкой качества? в) Ранняя остановка при $\eta = 10^{-3}$ остановилась примерно через 200 шагов. Какому коэффициенту штрафа L2 это соответствует по формуле из п. 7 и почему для Adam эта оценка только примерная?

*Ваш ответ:*

**Задание Р6. Сглаживание меток и сводная таблица.**

- `smooth_targets(y, num_classes, eps)` – сглаженные целевые распределения по п. 8 теории: `(1 - eps) * one_hot + eps / num_classes`, где `one_hot = F.one_hot(y, num_classes).float()`.
- `smoothed_cross_entropy(logits, y, eps)` – потеря $-\sum_k q_k \ln p_k$, усредненная по объектам; логарифмы вероятностей дает `F.log_softmax(logits, dim=1)`.

*Проверка по п. 8 теории.* Для класса 2 из четырех при $\varepsilon = 0{,}1$: $(0{,}025;\ 0{,}025;\ 0{,}925;\ 0{,}025)$. Логиты $(2, 0, 0, 0)$ и правильный класс 0: потеря $0{,}341$ без сглаживания и $0{,}491$ со сглаживанием. На случайных логитах ваша функция совпадает с `F.cross_entropy(..., label_smoothing=0.1)`. Затем готовые ячейки обучают сеть со сглаживанием, сеть со всеми способами сразу и выводят сводную таблицу.

In [ ]:
def smooth_targets(y, num_classes, eps):
    one_hot = F.one_hot(y, num_classes).float()
    return ...                                      # <-- (1 - eps) * one_hot + eps / K

def smoothed_cross_entropy(logits, y, eps):
    q = smooth_targets(y, logits.shape[1], eps)
    log_p = ...                                     # <-- логарифмы вероятностей по классам
    return ...                                      # <-- минус сумма q * log p по классам, среднее по объектам


print(show(smooth_targets(torch.tensor([2]), 4, 0.1)), '  ожидается [[0.025, 0.025, 0.925, 0.025]]')
z = torch.tensor([[2.0, 0.0, 0.0, 0.0]])
print(show(smoothed_cross_entropy(z, torch.tensor([0]), 0.0)), show(smoothed_cross_entropy(z, torch.tensor([0]), 0.1)),
      '  ожидается 0.341 0.491')
torch.manual_seed(RANDOM_STATE)
logits, y = torch.randn(64, 10), torch.randint(0, 10, (64,))
diff = (smoothed_cross_entropy(logits, y, 0.1) - F.cross_entropy(logits, y, label_smoothing=0.1)).abs().item()
print(diff < 1e-6, round(math.log(0.925 / 0.025), 3), '  ожидается True 3.611 (оптимальный разрыв логитов при K = 4)')

In [ ]:
# Готовая ячейка: сглаживание меток отдельно и все способы вместе
ls_loss = lambda logits, y: smoothed_cross_entropy(logits, y, 0.1)
train_variant('сглаживание меток 0.1', y_noisy, loss_fn=ls_loss)
train_variant('все вместе', y_noisy, p=0.5, wd=5, epochs=100, loss_fn=ls_loss, stopper=EarlyStopping(10))
train_variant('все вместе, чистые метки', y_clean, p=0.5, wd=5, epochs=100, loss_fn=ls_loss, stopper=EarlyStopping(10))

def confidence(model, X):
    """Средняя наибольшая вероятность – насколько сеть уверена в своих ответах."""
    model.eval()
    with torch.no_grad():
        return F.softmax(model(X), dim=1).max(dim=1).values.mean().item()

print(f'\nуверенность на валидации: без регуляризации {confidence(models["метки с ошибками"], X_val):.2f}, '
      f'со сглаживанием {confidence(models["сглаживание меток 0.1"], X_val):.2f}')
print('  ожидается: сглаживание само по себе не мешает запоминать ошибки (запомнено около 1.0), но снижает уверенность')
print('  примерно с 0.9 до 0.65 и валидационную потерю; «все вместе» – валидация около 0.77–0.79, запомнено меньше 0.1')

In [ ]:
# Готовая ячейка: сводная таблица и диаграмма
print(f'{"вариант":<26}{"эпох":>5}{"обуч.":>8}{"валид.":>8}{"разрыв":>8}{"потеря вал.":>13}{"тест":>7}{"запомнено":>11}')
for name, r in results.items():
    mem = f'{r["mem"]:.2f}' if r['noisy'] else '–'
    print(f'{name:<26}{r["epochs"]:>5}{r["train_acc"]:>8.3f}{r["val_acc"]:>8.3f}{r["gap"]:>8.3f}{r["val_loss"]:>13.3f}{r["test_acc"]:>7.3f}{mem:>11}')

noisy_names = [n for n, r in results.items() if r['noisy']]
fig, ax = plt.subplots(figsize=(8, 4.2))
vals = [results[n]['test_acc'] for n in noisy_names]
colors = [GRAY if n == 'метки с ошибками' else (NAVY if n == 'все вместе' else BLUE) for n in noisy_names]
ax.barh(noisy_names[::-1], vals[::-1], color=colors[::-1])
for i, v in enumerate(vals[::-1]):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0.6, 0.85); ax.set_xlabel('точность на тестовой выборке')
ax.set_title('1000 картинок, 200 меток испорчены', fontsize=10)
fig.tight_layout(); fig.savefig('summary.png', dpi=110); plt.close(fig)
display(Image('summary.png'))
print('  ожидается: без регуляризации тест около 0.71; лучшие варианты (снижение весов 10, прореживание 0.7,')
print('  ранняя остановка, все вместе) – около 0.74–0.79; на чистых метках тест почти не меняется (около 0.81),')
print('  но разрыв уменьшается примерно вдвое, а валидационная потеря – примерно с 1.0 до 0.6')

**Вопрос Р6.** а) Почему сглаживание меток само по себе не помешало сети запомнить испорченные метки? б) На испорченных метках регуляризация заметно подняла точность на тесте (в нашем запуске примерно с 0,71 до 0,78), а на правильных метках почти ничего не изменила. Что тогда она дала на правильных метках? Что нужно, чтобы поднять точность выше 0,81? в) Какие гиперпараметры в этой работе выбраны по валидационной выборке? Почему смотреть на тестовую выборку при выборе нельзя?

*Ваш ответ:*

### Задание повышенной сложности (к части 3Р)

1. **Неуверенность через многократное прореживание.** Возьмите сеть `models['прореживание 0.5']`, оставьте ее в режиме `train` и пропустите каждую тестовую картинку 50 раз. Средние вероятности дают ответ ансамбля подсетей, а разброс вероятности выбранного класса – оценку неуверенности (Гал и Гарамани, 2016). Сравните точность среднего ответа с ответом в режиме `eval`. Покажите, что у неверных ответов разброс в среднем больше, чем у верных, и что на картинке из случайного шума разброс еще больше. Сколько процентов ошибок можно отсеять, если отказываться отвечать на 10 % самых неуверенных картинок?
2. **Точные нули штрафа L1.** Обучите логистическую регрессию `nn.Linear(784, 10)` обычным SGD, а после каждого шага применяйте к весам мягкий порог $w \leftarrow \mathrm{sign}(w)\max(|w| - \eta\lambda,\ 0)$ (проксимальный шаг, формула 7.23 для одного шага). Сравните число точно нулевых весов с вариантом, где штраф L1 просто прибавлен к потере. Нарисуйте веса каждого класса картинкой 28 × 28.
3. **Ранняя остановка и штраф L2.** На линейной регрессии с двумя признаками разного масштаба (матрица $H$ с собственными значениями 10 и 0,1) нарисуйте траекторию градиентного спуска из нуля и кривую решений со штрафом L2 при разных $\lambda$ (рисунок 7.4 у Гудфеллоу). Проверьте численно, что точки траектории после $\tau$ шагов близки к решениям при $\lambda = 1/(\eta\tau)$.
4. **Баггинг.** Обучите 5 сетей `BigMLP` без регуляризации на бутстреп-выборках из 1000 картинок с испорченными метками (выборка с возвращением, раздел 7.11) и усредните их вероятности. Сравните точность ансамбля, отдельных сетей и одной сети с прореживанием, а также время обучения.
5. **Adam со штрафом против AdamW в обучении.** Для `torch.optim.Adam(weight_decay=λ)` и `torch.optim.AdamW(weight_decay=λ)` переберите по 5 значений $\lambda$ на логарифмической сетке, найдите лучшее значение для каждого по валидационной выборке и сравните лучшие точности на тестовой выборке. Постройте зависимость точности от $\lambda$ для обоих методов и объясните, почему сетки значений у них получились разными.

In [ ]:
# Место для задания повышенной сложности